### 3.3. Standardisierte Hyperparameter-Optimierung

In dieser Optimierungsrunde haben wir für alle Modelle gezielt Hyperparameter optimiert. Dabei wurden Parameter wie Baumanzahl, Tiefe, Lernrate und Regularisierung angepasst.

In der dritten Optimierungsrunde optimiere ich alle Modelle systematisch über ihre wichtigsten Parameter, um faire und vergleichbare Ergebnisse zu erzielen. Für jedes Modell wurden sinnvolle Wertebereiche definiert (z. B. Anzahl Bäume, Baumtiefe, Lernrate) und im Rahmen einer zufälligen Parameterauswahl getestet. Hierfür wurde eine eigene Implementierung eines Randomized Search verwendet, die es ermöglicht, gezielt unsere projektspezifische wirtschaftliche Zielfunktion als Bewertungsmaßstab zu verwenden. Diese Zielfunktion berücksichtigt die Kosten von Fehlern direkt, indem sie den Schaden durch False Negatives sowie Kosten für False Positives und Einsparungen durch True Positives abbildet. Ziel ist es, die vielversprechendsten Modelle für die abschließende Feinoptimierung auszuwählen.

Die Hyperparameteroptimierung selbst wurde bewusst ohne parallele Verarbeitung der Suchschleifen ausgeführt, um Stabilitätsprobleme (z. B. beim Pickling von Pipelines) zu vermeiden. Die interne Parallelisierung der Modelle wurde hingegen bei allen unterstützenden Algorithmen über n_jobs=4 aktiviert, um die Mehrkern-Architektur des eingesetzten Rechners effizient auszunutzen.

In [1]:
# Standard Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import logging

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay, 
    roc_auc_score, RocCurveDisplay, precision_recall_curve, PrecisionRecallDisplay
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# neue Modelle (nach pip install)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier, RUSBoostClassifier

# Logging & Warnings Setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
warnings.filterwarnings("ignore")

In [2]:
# Preprocessor laden
from pipeline.data_pipeline import preprocessor, X, y

## 1. Datenaufteilung

In [8]:

# damage extrahieren und aus X entfernen
damage_series = X['damage'].copy()
X = X.drop(columns=['damage'])

# stratify=y sorgt für eine proportionale Aufteilung der Klassen in Train/Test, wichtig bei unbalancierten Datensets.
if isinstance(y, pd.Series) and y.nunique() > 1 and y.notnull().all():
    stratify_param = y
else:
    stratify_param = None

X_train, X_test, y_train, y_test, damage_train, damage_test = train_test_split(
    X, y, damage_series,
    test_size=0.2,
    random_state=42,
    stratify=stratify_param
)


## 2. Zielfunktion definieren

In [9]:
def custom_objective_score(y_true, y_pred, damage_series):
    """
    Custom Zielfunktion:
    Score = -Summe Schaden der FN + (5 * TP) - (10 * FP)
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()
    
    fn_mask = (y_true == 1) & (y_pred == 0)
    sum_damage_fn = damage_series[fn_mask].sum()
    
    score = -sum_damage_fn + (5 * TP) - (10 * FP)
    return float(score)

## 3. Evaluationsfunktion definieren

In [10]:
def evaluate_model_with_curves(model, model_name, preprocessor, X_train, X_test, y_train, y_test, damage_test, threshold):
    logging.info(f"🔍 Starte Evaluation für Modell: {model_name}")

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    if y_train.isnull().any():
        logging.warning(f"y_train enthält NaN-Werte: {y_train.isnull().sum()} fehlende Werte!")
        return

    try:
        pipeline.fit(X_train, y_train)
    except Exception as e:
        logging.exception(f"Fehler beim Training: {e}")
        return

    has_proba = hasattr(model, "predict_proba")
    if has_proba:
        y_scores = pipeline.predict_proba(X_test)[:, 1]
        y_pred = (y_scores >= threshold).astype(int)
    else:
        logging.warning("⚠️ Modell unterstützt kein predict_proba.")
        y_scores = None
        y_pred = pipeline.predict(X_test)

    # Klassifikationsbericht
    print("\n📋 Klassifikationsbericht:")
    print(classification_report(y_test, y_pred, target_names=["NORMAL", "FRAUD"]))

    # Konfusionsmatrix
    cm = confusion_matrix(y_test, y_pred)
    print("\n🔲 Konfusionsmatrix:")
    print(cm)
    ConfusionMatrixDisplay(cm, display_labels=["NORMAL", "FRAUD"]).plot(cmap="Blues")
    plt.title(f"Konfusionsmatrix – {model_name} (Threshold: {threshold:.2f})")
    plt.show()

    # ROC & PR-Kurven
    if has_proba and y_scores is not None:
        auc = roc_auc_score(y_test, y_scores)
        logging.info(f"📈 ROC-AUC: {auc:.4f}")
        RocCurveDisplay.from_predictions(y_test, y_scores).plot()
        plt.plot([0, 1], [0, 1], 'k--', label='Zufall')
        plt.legend()
        plt.title(f"ROC-Kurve – {model_name}")
        plt.show()

        precision, recall, _ = precision_recall_curve(y_test, y_scores)
        PrecisionRecallDisplay(precision=precision, recall=recall).plot()
        plt.title(f"Precision-Recall-Kurve – {model_name}")
        plt.show()

    if len(y_test) != len(damage_test):
        logging.error("Länge von y_test und damage_test stimmen nicht überein!")
        return

    custom_score = custom_objective_score(y_test.reset_index(drop=True), y_pred, damage_test.reset_index(drop=True))
    print(f"\n🎯 Wert der Zielfunktion: {custom_score:.2f}")

    return custom_score  # Optionaler Rückgabewert

## 4. Randomized Search mit Custom Scoring

In [11]:
def randomized_search_with_custom_score(model_class, param_grid, preprocessor, 
                                        X, y, damage_series, 
                                        model_fixed_params=None, 
                                        n_iter=10, cv_splits=5, random_state=42):
    """
    Randomisierte Hyperparameter-Suche mit Custom Objective Function und optionalen festen Modellparametern.
    """
    rng = np.random.default_rng(seed=random_state)
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    best_score = -np.inf
    best_params = None
    param_grid_items = list(param_grid.items())

    for iteration in range(n_iter):
        params = {key: rng.choice(list(values)) for key, values in param_grid_items}
        full_params = {**params}
        if model_fixed_params:
            full_params.update(model_fixed_params)

        logging.info(f"\n➡️ Iteration {iteration+1}: teste Parameter {full_params}")
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            damage_valid = damage_series.iloc[valid_idx]

            pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', model_class(**full_params))
            ])

            try:
                pipeline.fit(X_train, y_train)
                y_pred = pipeline.predict(X_valid)
                score = custom_objective_score(
                    y_valid.reset_index(drop=True),
                    y_pred,
                    damage_valid.reset_index(drop=True)
                )
                fold_scores.append(score)
                logging.info(f"  Fold {fold} Score: {score:.2f}")
            except Exception as e:
                logging.error(f"❌ Fehler in Fold {fold}: {e}")
                continue

        if not fold_scores:
            logging.warning(f"⚠️ Keine gültigen Folds in Iteration {iteration+1}.")
            continue

        mean_score = np.mean(fold_scores)
        logging.info(f"  ➡️ Durchschnittlicher CV-Score: {mean_score:.2f}")

        if mean_score > best_score:
            best_score = mean_score
            best_params = full_params

    logging.info("\n✅ Beste Parameter gefunden!")
    logging.info(f"✅ Parameter: {best_params}")
    logging.info(f"✅ CV-Score: {best_score:.2f}")
    return best_params, best_score

## 5. Hyperparameter-Tuning-Setups für alle Modelle

In [12]:
# 1️⃣ Logistic Regression
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000]
}
randomized_search_with_custom_score(LogisticRegression, param_grid_lr, preprocessor, X_train, y_train, damage_train)

# 2️⃣ Random Forest
param_grid_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}
randomized_search_with_custom_score(RandomForestClassifier, param_grid_rf, preprocessor, X_train, y_train, damage_train)

# 3️⃣ Gradient Boosting
param_grid_gb = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0]
}
randomized_search_with_custom_score(GradientBoostingClassifier, param_grid_gb, preprocessor, X_train, y_train, damage_train)

# 4️⃣ HistGradientBoosting
param_grid_hgb = {
    'max_iter': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [None, 10, 20]
}
randomized_search_with_custom_score(HistGradientBoostingClassifier, param_grid_hgb, preprocessor, X_train, y_train, damage_train)

# 5️⃣ XGBoost
param_grid_xgb = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}
randomized_search_with_custom_score(
    XGBClassifier, 
    param_grid_xgb, 
    preprocessor, 
    X_train, 
    y_train, 
    damage_train,
    model_fixed_params={'use_label_encoder': False, 'eval_metric': 'logloss', 'n_jobs': 4}
)

# 6️⃣ LightGBM
param_grid_lgbm = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [20, 31, 50],
    'max_depth': [5, 10, 20]
}
randomized_search_with_custom_score(
    LGBMClassifier, 
    param_grid_lgbm, 
    preprocessor, 
    X_train, 
    y_train, 
    damage_train, 
    model_fixed_params={'n_jobs': 4}
)

# 7️⃣ CatBoost
param_grid_cat = {
    'iterations': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'depth': [4, 6, 8]
}
randomized_search_with_custom_score(
    CatBoostClassifier, 
    param_grid_cat, 
    preprocessor, 
    X_train, 
    y_train, 
    damage_train, 
    model_fixed_params={'verbose': 0}
)

# 8️⃣ BalancedRandomForest
param_grid_brf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [5, 10, None]
}
randomized_search_with_custom_score(
    BalancedRandomForestClassifier, 
    param_grid_brf, 
    preprocessor, 
    X_train, 
    y_train, 
    damage_train, 
    model_fixed_params={'n_jobs': 4}
)

# 9️⃣ RUSBoostClassifier
param_grid_rusboost = {
    'n_estimators': [50, 100, 300],
    'learning_rate': [0.01, 0.05, 0.1]
}

randomized_search_with_custom_score(
    RUSBoostClassifier, 
    param_grid_rusboost, 
    preprocessor, 
    X_train, 
    y_train, 
    damage_train, 
    model_fixed_params={'estimator': DecisionTreeClassifier(max_depth=3), 'random_state': 42}
)

2025-06-02 23:10:46,970 - INFO - 
➡️ Iteration 1: teste Parameter {'C': 0.01, 'solver': 'lbfgs', 'max_iter': 1000}
2025-06-02 23:11:08,812 - INFO -   Fold 1 Score: -6272.90
2025-06-02 23:11:13,784 - INFO -   Fold 2 Score: -6401.57
2025-06-02 23:11:18,097 - INFO -   Fold 3 Score: -6438.78
2025-06-02 23:11:22,518 - INFO -   Fold 4 Score: -5863.19
2025-06-02 23:11:27,325 - INFO -   Fold 5 Score: -6439.45
2025-06-02 23:11:27,325 - INFO -   ➡️ Durchschnittlicher CV-Score: -6283.18
2025-06-02 23:11:27,325 - INFO - 
➡️ Iteration 2: teste Parameter {'C': 10.0, 'solver': 'liblinear', 'max_iter': 1000}
2025-06-02 23:11:34,255 - INFO -   Fold 1 Score: -6115.07
2025-06-02 23:11:41,101 - INFO -   Fold 2 Score: -6276.28
2025-06-02 23:11:47,900 - INFO -   Fold 3 Score: -6245.61
2025-06-02 23:11:54,403 - INFO -   Fold 4 Score: -5727.94
2025-06-02 23:12:01,404 - INFO -   Fold 5 Score: -6383.17
2025-06-02 23:12:01,404 - INFO -   ➡️ Durchschnittlicher CV-Score: -6149.61
2025-06-02 23:12:01,419 - INFO - 


[LightGBM] [Info] Number of positive: 3200, number of negative: 96442
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004782 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2195
[LightGBM] [Info] Number of data points in the train set: 99642, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032115 -> initscore=-3.405791
[LightGBM] [Info] Start training from score -3.405791


2025-06-03 02:30:34,095 - INFO -   Fold 1 Score: -5233.61


[LightGBM] [Info] Number of positive: 3200, number of negative: 96442
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007479 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2197
[LightGBM] [Info] Number of data points in the train set: 99642, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032115 -> initscore=-3.405791
[LightGBM] [Info] Start training from score -3.405791


2025-06-03 02:30:38,952 - INFO -   Fold 2 Score: -4913.75


...

[LightGBM] [Info] Number of positive: 3200, number of negative: 96443
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007520 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2201
[LightGBM] [Info] Number of data points in the train set: 99643, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032115 -> initscore=-3.405801
[LightGBM] [Info] Start training from score -3.405801


2025-06-03 02:35:05,897 - INFO -   Fold 4 Score: -4958.05


[LightGBM] [Info] Number of positive: 3200, number of negative: 96443
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006263 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2195
[LightGBM] [Info] Number of data points in the train set: 99643, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032115 -> initscore=-3.405801
[LightGBM] [Info] Start training from score -3.405801


2025-06-03 02:35:12,210 - INFO -   Fold 5 Score: -5146.88
2025-06-03 02:35:12,211 - INFO -   ➡️ Durchschnittlicher CV-Score: -5062.25
2025-06-03 02:35:12,212 - INFO - 
✅ Beste Parameter gefunden!
2025-06-03 02:35:12,212 - INFO - ✅ Parameter: {'n_estimators': 100, 'learning_rate': 0.05, 'num_leaves': 50, 'max_depth': 5, 'n_jobs': 4}
2025-06-03 02:35:12,213 - INFO - ✅ CV-Score: -4901.73
2025-06-03 02:35:12,230 - INFO - 
➡️ Iteration 1: teste Parameter {'iterations': 100, 'learning_rate': 0.1, 'depth': 6, 'verbose': 0}
2025-06-03 02:35:19,447 - INFO -   Fold 1 Score: -5095.48
2025-06-03 02:35:26,376 - INFO -   Fold 2 Score: -4775.43
2025-06-03 02:35:33,286 - INFO -   Fold 3 Score: -4805.88
2025-06-03 02:35:40,129 - INFO -   Fold 4 Score: -4643.18
2025-06-03 02:35:47,054 - INFO -   Fold 5 Score: -4917.79
2025-06-03 02:35:47,055 - INFO -   ➡️ Durchschnittlicher CV-Score: -4847.55
2025-06-03 02:35:47,056 - INFO - 
➡️ Iteration 2: teste Parameter {'iterations': 300, 'learning_rate': 0.05, 'de

({'n_estimators': 50,
  'learning_rate': 0.01,
  'estimator': DecisionTreeClassifier(max_depth=3),
  'random_state': 42},
 -20457.448)

## 6. Master-Liste mit allen Modellen und Parametern:

In [ ]:
model_configs = [
    {
        'name': 'LogisticRegression',
        'model_class': LogisticRegression,
        'param_grid': {
            'C': [0.01, 0.1, 1, 10, 100],
            'solver': ['liblinear', 'lbfgs'],
            'max_iter': [1000]
        },
        'fixed_params': {}
    },
    {
        'name': 'RandomForestClassifier',
        'model_class': RandomForestClassifier,
        'param_grid': {
            'n_estimators': [100, 300, 500],
            'max_depth': [5, 10, 20, None],
            'min_samples_split': [2, 5, 10]
        },
        'fixed_params': {'n_jobs': 4}
    },
    {
        'name': 'GradientBoostingClassifier',
        'model_class': GradientBoostingClassifier,
        'param_grid': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8, 1.0]
        },
        'fixed_params': {}
    },
    {
        'name': 'HistGradientBoostingClassifier',
        'model_class': HistGradientBoostingClassifier,
        'param_grid': {
            'max_iter': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [None, 10, 20]
        },
        'fixed_params': {}
    },
    {
        'name': 'XGBClassifier',
        'model_class': XGBClassifier,
        'param_grid': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0]
        },
        'fixed_params': {'use_label_encoder': False, 'eval_metric': 'logloss', 'n_jobs': 4}
    },
    {
        'name': 'LGBMClassifier',
        'model_class': LGBMClassifier,
        'param_grid': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [20, 31, 50],
            'max_depth': [5, 10, 20]
        },
        'fixed_params': {'n_jobs': 4}
    },
    {
        'name': 'CatBoostClassifier',
        'model_class': CatBoostClassifier,
        'param_grid': {
            'iterations': [100, 300, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'depth': [4, 6, 8]
        },
        'fixed_params': {'verbose': 0}
    },
    {
        'name': 'BalancedRandomForestClassifier',
        'model_class': BalancedRandomForestClassifier,
        'param_grid': {
            'n_estimators': [100, 300, 500],
            'max_depth': [5, 10, None]
        },
        'fixed_params': {'n_jobs': 4}
    },
    {
        'name': 'RUSBoostClassifier',
        'model_class': RUSBoostClassifier,
        'param_grid': {
            'n_estimators': [50, 100, 300],
            'learning_rate': [0.01, 0.05, 0.1]
        },
        'fixed_params': {'estimator': DecisionTreeClassifier(max_depth=3), 'random_state': 42}
    }
]

## 7. Ausgabe aller Ergebnisse

In [ ]:
# Speicher für alle Ergebnisse
results = []

# Master-Loop
for config in model_configs:
    print(f"\n\n==========================")
    print(f"🔍 Tune: {config['name']}")

    best_params, best_score = randomized_search_with_custom_score(
        model_class=config['model_class'],
        param_grid=config['param_grid'],
        preprocessor=preprocessor,
        X=X_train,
        y=y_train,
        damage_series=damage_train,
        n_iter=10,
        cv_splits=5,
        model_fixed_params=config['fixed_params']
    )
    
    # Merge fixed + best params
    full_params = {**best_params, **config['fixed_params']}

    # Train final model
    try:
        final_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', config['model_class'](**full_params, random_state=42))
        ])
    except TypeError:
        # Falls random_state nicht unterstützt wird
        final_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', config['model_class'](**full_params))
        ])

    final_pipeline.fit(X_train, y_train)

    # Evaluate on testset
    evaluate_model_with_curves(
        model=final_pipeline.named_steps['classifier'],
        model_name=config['name'] + " (optimiert)",
        preprocessor=preprocessor,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        damage_test=damage_test,
        threshold=0.5
    )

    # Ergebnis speichern
    results.append({
        'Model': config['name'],
        'Best_Params': full_params,
        'CV_Custom_Score': best_score
    })

# Zusammenfassung als DataFrame:
results_df = pd.DataFrame(results)
print("\n\n📊 Zusammenfassung aller Modelle:")
print(results_df)


## 8. Threshold-Optimierer

In [ ]:
def find_best_threshold(model, preprocessor, X_test, y_test, damage_test, thresholds=np.arange(0.01, 1.0, 0.01)):
    """
    Findet den besten Threshold, der den Custom Objective Score maximiert.
    """
    # Pipeline für Preprocessing
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Fit auf Gesamtdaten (hier wird idR vorher schon fit sein - zur Sicherheit)
    pipeline.fit(X_train, y_train)

    # Prüfen, ob Modell Wahrscheinlichkeiten liefert
    if not hasattr(model, "predict_proba"):
        print("⚠️ Dieses Modell unterstützt kein predict_proba. Threshold-Optimierung nicht möglich.")
        return None, None

    y_scores = pipeline.predict_proba(X_test)[:, 1]

    best_threshold = 0.5
    best_score = -np.inf

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        score = custom_objective_score(
            y_test.reset_index(drop=True),
            y_pred,
            damage_test.reset_index(drop=True)
        )
        if score > best_score:
            best_score = score
            best_threshold = thresh

    print(f"🎯 Optimaler Threshold: {best_threshold:.2f} mit Custom Score: {best_score:.2f}")
    return best_threshold, best_score


## 9. Auto-Integration der Threshold-Optimierung für alle Modelle

In [ ]:
results = []

for config in model_configs:
    print(f"\n\n==========================")
    print(f"🔍 Tune: {config['name']}")

    # 1️⃣ Randomized Search
    best_params, best_cv_score = randomized_search_with_custom_score(
        model_class=config['model_class'],
        param_grid=config['param_grid'],
        preprocessor=preprocessor,
        X=X_train,
        y=y_train,
        damage_series=damage_train,
        n_iter=10,
        cv_splits=5,
        model_fixed_params=config['fixed_params']
    )
    
    # 2️⃣ Merge der Parameter
    full_params = {**best_params, **config['fixed_params']}

    # 3️⃣ Train final model
    try:
        final_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', config['model_class'](**full_params, random_state=42))
        ])
    except TypeError:
        final_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', config['model_class'](**full_params))
        ])

    final_pipeline.fit(X_train, y_train)

    # 4️⃣ Threshold-Optimierung
    best_threshold, best_test_score = find_best_threshold(
        model=final_pipeline.named_steps['classifier'],
        preprocessor=preprocessor,
        X_test=X_test,
        y_test=y_test,
        damage_test=damage_test
    )

    # 5️⃣ Ergebnisse speichern
    results.append({
        'Model': config['name'],
        'Best_Params': full_params,
        'CV_Custom_Score': best_cv_score,
        'Best_Threshold': best_threshold,
        'Test_Custom_Score': best_test_score
    })

# 6️⃣ Zusammenfassung ausgeben
results_df = pd.DataFrame(results)
print("\n\n📊 Zusammenfassung aller Modelle inkl. Threshold-Optimierung:")
print(results_df)
